# 31depth — Encoder-depth × expert-depth factorial

Runs the complete paired 2×2 factorial across seeds 0–2: the Notebook 31 baseline, deeper experts only, deeper shared encoder only, and both deeper. Width is held at 128 in the added encoder layer, the gate remains linear, and every training/evaluation setting is otherwise fixed. Compatible Stage-A checkpoints are reused without sharing any stage whose architecture changed.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDIES = [
    ('shallow_linear', 'config/compact_soft_moe_3seed.yaml', 'nfv3_4way_moe_soft_comparable_repro_v1'),
    ('shallow_deep_expert', 'config/compact_soft_moe_depth_expert_3seed.yaml', 'nfv3_4way_moe_soft_depth_expert_v1'),
    ('deep_encoder_linear', 'config/compact_soft_moe_depth_encoder_3seed.yaml', 'nfv3_4way_moe_soft_depth_encoder_v1'),
    ('deep_encoder_deep_expert', 'config/compact_soft_moe_depth_both_3seed.yaml', 'nfv3_4way_moe_soft_depth_both_v1'),
]
EXECUTE = False
MAX_NEW_SEEDS_PER_STUDY = 3  # Set 1 to advance every condition one seed per session.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Complete factorial dry-run and execution

The order is intentional: baseline Stage A is available before the deeper-expert cell, and deep-encoder Stage A is available before the combined cell. Every command is resumable.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_targeted_depth_search.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    model = study['backbone']['model']
    print(f'=== {condition}: encoder={model["encoder"]["hidden_dims"]}, experts={model["expert"]["hidden_dims"]}, parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31depth-{condition}-dry-run')
    if EXECUTE:
        launch = [*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)]
        run_streaming_logged(launch, str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'31depth-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review the graph, set EXECUTE=True, and rerun this cell.')

## Factorial effects on validation and locked test

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
if all(complete.values()):
    validation = []; test = []; classes = []; resources = []
    for name, path in summaries.items():
        for filename, target in [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Resource_Accounting.csv', resources)]:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    print('=== Validation guardrails ==='); display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    for name in pivot.columns:
        if name != 'shallow_linear': pivot[f'delta__{name}'] = pivot[name] - pivot['shallow_linear']
    print('=== Paired validation effects ==='); display(pivot)
    cell_means = validation.groupby('condition')['validation_macro_f1'].mean()
    interaction = cell_means['deep_encoder_deep_expert'] - cell_means['deep_encoder_linear'] - cell_means['shallow_deep_expert'] + cell_means['shallow_linear']
    print('Mean encoder main effect:', ((cell_means['deep_encoder_linear'] + cell_means['deep_encoder_deep_expert']) - (cell_means['shallow_linear'] + cell_means['shallow_deep_expert'])) / 2)
    print('Mean expert main effect:', ((cell_means['shallow_deep_expert'] + cell_means['deep_encoder_deep_expert']) - (cell_means['shallow_linear'] + cell_means['deep_encoder_linear'])) / 2)
    print('Encoder × expert interaction:', interaction)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    print('=== Locked-test descriptive mean and SD ==='); display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Per-class F1 mean ==='); display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    print('=== Resource accounting ==='); display(pd.concat(resources, ignore_index=True))
else: print('Factorial tables appear after all four three-seed manifests are complete.')

## Training and latent-space diagnostics

In [ ]:
if all(complete.values()):
    histories = []; latent = []
    for name, path in summaries.items():
        frame = pd.read_csv(path / 'Training_History_Summary_3Seed.csv'); frame.insert(0, 'condition', name); histories.append(frame)
        frame = pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'); frame.insert(0, 'condition', name); latent.append(frame)
    display(pd.concat(histories, ignore_index=True)); display(pd.concat(latent, ignore_index=True))
    for name, path in summaries.items():
        print('===', name, 'cross-seed curves ===')
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))